# Preprocessed Big Data - VTuber 1B (Kaggle)

VTuber 1B is a collection of live chats, and stream events for Virtual Youtubers (VTubers) such as subscription, super chats, chat deletion events, banning events, etc. The upcoming analytics will only focus on the `channels.csv` and  **32 gigabytes of parquet files** for year 2021 chats. 

The challenges:

__For the Chats `chats_2021-*.parquet`__:
1. Records started on Jan 15, 2021 only. 
2. December 2021 parquet file is corrupted, unable to load
3. Schema for the scoped live chat files are not uniform.
4. The column `isMember` has to be dropped as it is NULL as per [Documentation](https://github.com/sigvt/vtuber-livechat-dataset)

__For the Channels `channels.csv`__:
1. The column `group` has to be dropped as it is **68% NULL**
2. `subscriptionCount` column will not be part of the query to avoid biases and `photo` column would be an irrelevant data.

In [1]:
import glob
import gc
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("VTuber_ETL_16GB_Fix")
    .master("local[2]") 
    # FIX 1: Allocate the bulk of the 16GB directly to the Driver
    .config("spark.driver.memory", "10g")

    # FIX 2: Force the JVM to use the G1 Garbage Collector and spam triggers
    .config(
        "spark.driver.extraJavaOptions",
        "-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35"
    )
    # Keep the read-buffer bite sizes small (64MB)
    .config("spark.sql.files.maxPartitionBytes", "67108864") 
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

target_columns = [
    F.col("timestamp").cast("timestamp"),
    F.col("authorChannelId").cast("string"),
    F.col("videoId").cast("string"),
    F.col("channelId").cast("string"),
    F.col("bodyLength").cast("long")
]

In [2]:
# Note: Adjust base_path if needed based on your environment
base_path = '/mnt/data/public/vtuber/chats_2021-*.parquet'
all_files = sorted(glob.glob(base_path))
safe_files = [f for f in all_files if '2021-12' not in f]

unified_path = '/mnt/processed/private/msds2026/slt9a/unified_chats_2021'

for file_path in safe_files:
    print(f"Processing: {file_path}")
    # Load and standardize
    df_month = spark.read.parquet(file_path).select(target_columns)
    # Write to disk
    df_month.write.mode("append").parquet(unified_path)
    # Clean up DataFrame from Spark memory
    df_month.unpersist()
    del df_month
    # Clean up Python objects
    gc.collect() 
    # Explicitly clear Spark's SQL cache just to be safe
    spark.catalog.clearCache()

print("ETL Complete! All months merged without crashing.")

Processing: /mnt/data/public/vtuber/chats_2021-01.parquet
Processing: /mnt/data/public/vtuber/chats_2021-02.parquet
Processing: /mnt/data/public/vtuber/chats_2021-03.parquet
Processing: /mnt/data/public/vtuber/chats_2021-04.parquet
Processing: /mnt/data/public/vtuber/chats_2021-05.parquet
Processing: /mnt/data/public/vtuber/chats_2021-06.parquet
Processing: /mnt/data/public/vtuber/chats_2021-07.parquet
Processing: /mnt/data/public/vtuber/chats_2021-08.parquet
Processing: /mnt/data/public/vtuber/chats_2021-09.parquet
Processing: /mnt/data/public/vtuber/chats_2021-10.parquet
Processing: /mnt/data/public/vtuber/chats_2021-11.parquet
ETL Complete! All months merged without crashing.


In [6]:
import pandas as pd

chdf = pd.read_csv('/mnt/data/public/vtuber/channels.csv')
print(chdf.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   channelId          1358 non-null   object
 1   name               1358 non-null   object
 2   englishName        1358 non-null   object
 3   affiliation        1358 non-null   object
 4   group              429 non-null    object
 5   subscriptionCount  1358 non-null   int64 
 6   videoCount         1358 non-null   int64 
 7   photo              1358 non-null   object
dtypes: int64(2), object(6)
memory usage: 85.0+ KB
None


In [7]:
chdf = chdf.drop(
    ['group', 'subscriptionCount', 'photo'],
    axis=1
)

chdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   channelId    1358 non-null   object
 1   name         1358 non-null   object
 2   englishName  1358 non-null   object
 3   affiliation  1358 non-null   object
 4   videoCount   1358 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 53.2+ KB


In [8]:
chdf.to_csv('/mnt/processed/private/'\
            'msds2026/slt9a/channels_2021.csv')